# Hybrid Anime Recommendation System - Demo

This notebook demonstrates how to use the Anime Recommendation System.

In [ ]:
import sys
sys.path.append('..')

from preprocessing import DataLoader, MatrixBuilder
from models.content import ContentBasedRecommender
from models.collaborative import MatrixFactorization
from models.popularity import PopularityModel
from models.hybrid import HybridEngine
from evaluation import RecommenderMetrics

## 1. Load Data

In [ ]:
# Load datasets
loader = DataLoader()
anime_df = loader.get_merged_anime_data()
print(f"Anime: {len(anime_df)} records")
anime_df.head()

In [ ]:
# Load ratings (sampled)
ratings_df = loader.load_ratings(sample=True)
print(f"Ratings: {len(ratings_df):,} records")
ratings_df.head()

## 2. Content-Based Recommendations

In [ ]:
# Train content-based model
content_model = ContentBasedRecommender()
content_model.fit(anime_df, use_sbert=False)  # Skip SBERT for speed

In [ ]:
# Get similar anime to "Naruto"
similar = content_model.get_similar_anime("Naruto", top_k=10, method="tfidf")
for rec in similar:
    print(f"{rec['name']} - Score: {rec['score']}, Similarity: {rec['similarity']:.4f}")

In [ ]:
# Search for anime
results = content_model.search_anime("death", top_k=5)
for r in results:
    print(f"{r['name']} ({r['type']}) - Score: {r['score']}")

## 3. Collaborative Filtering

In [ ]:
# Build user-item matrix
builder = MatrixBuilder()
builder.build_rating_matrix(ratings_df)
print(f"Matrix: {builder.n_users:,} users x {builder.n_items:,} items")

In [ ]:
# Train Matrix Factorization
mf_model = MatrixFactorization(n_factors=50, n_epochs=10)
mf_model.fit(
    builder.user_item_matrix,
    builder.anime_to_idx,
    builder.idx_to_anime,
    builder.user_to_idx,
    builder.idx_to_user
)

In [ ]:
# Get recommendations for a user
test_user = list(builder.user_to_idx.keys())[0]
recs = mf_model.recommend_for_user(test_user, top_k=10)
print(f"Recommendations for user {test_user}:")
for rec in recs:
    print(f"  Anime {rec['mal_id']}: predicted rating = {rec['predicted_rating']:.2f}")

## 4. Popularity-Based Recommendations

In [ ]:
# Train popularity model
pop_model = PopularityModel()
pop_model.fit(anime_df, ratings_df)

In [ ]:
# Top rated anime
print("=== Top Rated ===")
for anime in pop_model.get_top_rated(5):
    print(f"  {anime['name']} - Score: {anime['score']}")

In [ ]:
# Filter by genre
print("=== Top Action Anime ===")
for anime in pop_model.get_top_rated(5, genre="Action"):
    print(f"  {anime['name']} - Score: {anime['score']}")

## 5. Hybrid Engine

In [ ]:
# Create hybrid engine
hybrid = HybridEngine(
    content_model=content_model,
    collaborative_model=mf_model,
    popularity_model=pop_model
)
hybrid.set_anime_info(anime_df)

In [ ]:
# Hybrid recommendations
print("=== Hybrid Recommendations for 'Death Note' ===")
recs = hybrid.recommend_similar_anime("Death Note", top_k=10)
for rec in recs:
    print(f"  {rec['name']} - Score: {rec['score']}, Hybrid: {rec['hybrid_score']:.4f}")

In [ ]:
# Update weights
hybrid.set_weights({
    'content': 0.5,
    'collaborative': 0.3,
    'popularity': 0.2
})
print(f"New weights: {hybrid.weights}")

## 6. Evaluation

In [ ]:
# Sample evaluation
recommended = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
relevant = {2, 4, 6, 8, 10}

print(f"Precision@5: {RecommenderMetrics.precision_at_k(recommended, relevant, 5):.4f}")
print(f"Recall@10: {RecommenderMetrics.recall_at_k(recommended, relevant, 10):.4f}")
print(f"NDCG@10: {RecommenderMetrics.ndcg_at_k(recommended, relevant, 10):.4f}")

## 7. Save/Load Models

In [ ]:
# Save hybrid engine
from config import MODELS_DIR
hybrid.save(MODELS_DIR / "hybrid_demo")

In [ ]:
# Load hybrid engine
loaded_hybrid = HybridEngine()
loaded_hybrid.load(MODELS_DIR / "hybrid_demo")

# Test
recs = loaded_hybrid.recommend_similar_anime("Naruto", top_k=3)
for rec in recs:
    print(f"  {rec['name']}")